In [1]:
import pandas as pd
import numpy as np
import json
import base64
from typing import Dict, List
import os
from google import genai
from google.genai import types
import anthropic
import openai

In [ ]:
def create_image(filepath: str) -> str:
    with open(filepath, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

In [3]:
def create_word_data(word: str, image_paths: List[str]) -> Dict:
    return {
        "word": word,
        "images": image_paths
    }

In [4]:
def get_prompt(prompt_type: str, word: str) -> str:
    
    
    if prompt_type == "minimal":
        return f"""You will see 5 images where a native speaker used the word "{word}".
                    What does "{word}" mean?.
                    Answer with ONE WORD in English."""

    elif prompt_type == "elaborate":
        return f"""You are a linguist documenting an unknown language. You have no prior knowledge of how these speakers categorize reality.

A native speaker used the word "{word}" when pointing at objects in these 5 situations.

Based ONLY on observable patterns in the images - without assuming any framework for categorizing reality - what does "{word}" most likely mean?



Answer with ONE WORD in English."""

    else:
        raise ValueError(f"Unknown prompt_type: {prompt_type}. Use 'minimal' or 'elaborate'")

In [5]:
google_api_key = os.getenv("GOOGLE_API_KEY")

def call_gemini(prompt: str, images: List[str], api_key: str, 
                model: str = "gemini-2.5-flash") -> str:
    client = genai.Client(api_key=api_key)
    
    contents = [prompt]
    for img_data in images:
        contents.append({
            "inline_data": {"mime_type": "image/jpeg", "data": img_data}
        })
    
    response = client.models.generate_content(
        model=model,
        contents=contents
    )
    
    return response.text.strip()

In [6]:
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

def call_claude(prompt: str, images: List[str], api_key: str,
                model: str = "claude-sonnet-4-5-20250929") -> str:
    client = anthropic.Anthropic(api_key=api_key)
    
    content = []
    for img_data in images:
        content.append({
            "type": "image",
            "source": {
                "type": "base64",
                "media_type": "image/jpeg",
                "data": img_data
            }
        })
    content.append({"type": "text", "text": prompt})
    
    response = client.messages.create(
        model=model,
        max_tokens=300,
        messages=[{"role": "user", "content": content}]
    )
    
    return response.content[0].text.strip()

In [7]:
openai_api_key = os.getenv("OPENAI_API_KEY")

def call_openai(prompt: str, images: List[str], api_key: str,
                model: str = "gpt-4o") -> str:
    client = openai.OpenAI(api_key=api_key)
    
    content = [{"type": "text", "text": prompt}]
    for img_data in images:
        content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{img_data}"}
        })
    
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": content}],
        max_tokens=300
    )
    
    return response.choices[0].message.content.strip()

In [8]:
def run_experiment(
    word_data: Dict,
    model_name: str,
    api_key: str,
    prompt_type: str,
    n_iterations: int = 10,
) -> Dict:
   
    word = word_data["word"]
    image_paths = word_data["images"]
    
    print(f"Encoding {len(image_paths)} images for '{word}'...")
    encoded_images = [create_image(path) for path in image_paths]
    
    prompt = get_prompt(prompt_type, word)
    
    api_functions = {
        "gemini": call_gemini,
        "claude": call_claude,
        "gpt4": call_openai
    }
    
    if model_name not in api_functions:
        raise ValueError(f"Unknown model: {model_name}. Use 'gemini', 'claude', or 'gpt4'")
    
    api_call = api_functions[model_name]
    
    results = []
    print(f"\nRunning {n_iterations} iterations for '{word}' with {model_name} ({prompt_type} prompt)...")
    
    for i in range(n_iterations):
        try:
            response = api_call(prompt, encoded_images, api_key)
            results.append(response)
            print(f"  Trial {i+1}/{n_iterations}: {response}")
            
        except Exception as e:
            print(f"  Trial {i+1}/{n_iterations}: ERROR - {e}")
            results.append(f"ERROR: {e}")
    
    return {
        "word": word,
        "model": model_name,
        "prompt_type": prompt_type,
        "n_iterations": n_iterations,
        "responses": results}

In [9]:
julpa_data = create_word_data(word ='julpa', image_paths =  ['main_quine_file/quine_images/book_images/single_page_from_a_book.jpg', 
                                              'main_quine_file/quine_images/book_images/person_reading_a_book.jpg',
                                             'main_quine_file/quine_images/book_images/open_book.jpg',
                                             'main_quine_file/quine_images/book_images/coloring_book.jpg',
                                             'main_quine_file/quine_images/book_images/closed_hard_cover_book.jpg'])


run_experiment(word_data = julpa_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'julpa'...

Running 10 iterations for 'julpa' with gemini (minimal prompt)...
  Trial 1/10: Book
  Trial 2/10: Book
  Trial 3/10: Book
  Trial 4/10: Book
  Trial 5/10: Book
  Trial 6/10: Book
  Trial 7/10: Book
  Trial 8/10: Book
  Trial 9/10: Book
  Trial 10/10: Book


{'word': 'julpa',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Book',
  'Book',
  'Book',
  'Book',
  'Book',
  'Book',
  'Book',
  'Book',
  'Book',
  'Book']}

In [10]:
run_experiment(word_data = julpa_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'julpa'...

Running 10 iterations for 'julpa' with gemini (elaborate prompt)...
  Trial 1/10: Book
  Trial 2/10: BOOK
  Trial 3/10: BOOK
  Trial 4/10: BOOK
  Trial 5/10: BOOK
  Trial 6/10: Book
  Trial 7/10: BOOK
  Trial 8/10: BOOK
  Trial 9/10: BOOK
  Trial 10/10: BOOK


{'word': 'julpa',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Book',
  'BOOK',
  'BOOK',
  'BOOK',
  'BOOK',
  'Book',
  'BOOK',
  'BOOK',
  'BOOK',
  'BOOK']}

In [11]:
run_experiment(word_data = julpa_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'julpa'...

Running 10 iterations for 'julpa' with claude (minimal prompt)...
  Trial 1/10: Looking at the images, I can see:
1. An open book with text
2. A person reading a book
3. An open book on a table
4. A coloring book with pencils
5. A closed book on a table

The word "julpa" appears to mean: **book**
  Trial 2/10: Looking at the images, I can see:
1. An open book with regular text
2. A person reading a book
3. An open book showing pages
4. A coloring book with illustrations
5. A closed book with brown cover

The word "julpa" appears in the context of these various books being shown. Based on the images displaying different types and states of books, "julpa" means:

**book**
  Trial 3/10: Looking at the images, I can see:
1. A page from an old book with text
2. A person reading a book
3. An open book on a table
4. A coloring book with pencils
5. A closed book with a brown cover

The word "julpa" appears to mean: **book**
  Trial 4/10: Looking at the images,

{'word': 'julpa',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Looking at the images, I can see:\n1. An open book with text\n2. A person reading a book\n3. An open book on a table\n4. A coloring book with pencils\n5. A closed book on a table\n\nThe word "julpa" appears to mean: **book**',
  'Looking at the images, I can see:\n1. An open book with regular text\n2. A person reading a book\n3. An open book showing pages\n4. A coloring book with illustrations\n5. A closed book with brown cover\n\nThe word "julpa" appears in the context of these various books being shown. Based on the images displaying different types and states of books, "julpa" means:\n\n**book**',
  'Looking at the images, I can see:\n1. A page from an old book with text\n2. A person reading a book\n3. An open book on a table\n4. A coloring book with pencils\n5. A closed book with a brown cover\n\nThe word "julpa" appears to mean: **book**',
  'Looking at the images, I can see:\n1. 

In [12]:
run_experiment(word_data = julpa_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'julpa'...

Running 10 iterations for 'julpa' with claude (elaborate prompt)...
  Trial 1/10: # Analysis

Let me examine what's observable in each image where "julpa" was used:

1. **Image 1**: Open book with visible text pages
2. **Image 2**: Person holding an open book
3. **Image 3**: Open book standing on a surface
4. **Image 4**: Open book/notebook with drawings and colored pencils
5. **Image 5**: Closed book on a surface

**Observable pattern**: In all five situations, there is a book present. 

- Images 1, 2, 3, and 4 show books that are **open**
- Image 5 shows a book that is **closed**

The common element across ALL images is the presence of a **book** (regardless of state - open or closed, with text or drawings, being held or resting on surface).

**Answer**: BOOK
  Trial 2/10: # Analysis

Looking at the 5 images where "julpa" was used:

1. Open book with visible text pages
2. Person holding and reading an open book
3. Open book standing on its pages
4. O

{'word': 'julpa',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['# Analysis\n\nLet me examine what\'s observable in each image where "julpa" was used:\n\n1. **Image 1**: Open book with visible text pages\n2. **Image 2**: Person holding an open book\n3. **Image 3**: Open book standing on a surface\n4. **Image 4**: Open book/notebook with drawings and colored pencils\n5. **Image 5**: Closed book on a surface\n\n**Observable pattern**: In all five situations, there is a book present. \n\n- Images 1, 2, 3, and 4 show books that are **open**\n- Image 5 shows a book that is **closed**\n\nThe common element across ALL images is the presence of a **book** (regardless of state - open or closed, with text or drawings, being held or resting on surface).\n\n**Answer**: BOOK',
  '# Analysis\n\nLooking at the 5 images where "julpa" was used:\n\n1. Open book with visible text pages\n2. Person holding and reading an open book\n3. Open book standing on its pages\n

In [13]:
run_experiment(word_data = julpa_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'julpa'...

Running 10 iterations for 'julpa' with gpt4 (minimal prompt)...
  Trial 1/10: Sorry, I can't determine the meaning of "julpa" from these images. Could you provide more context?
  Trial 2/10: Sorry, I can't determine what "julpa" means from the images provided.
  Trial 3/10: I'm sorry, I can't determine the meaning of "julpa" from the images. Can you provide more context or a sentence using the word?
  Trial 4/10: Sorry, I can't determine the meaning of "julpa" from the images alone. If you provide more context or text, I might be able to help!
  Trial 5/10: I'm sorry, I can't help with that.
  Trial 6/10: It’s unclear from the provided images what "julpa" means, as there are no contextual clues or text visible related to this term. Could you provide more context or an excerpt from the text where "julpa" is used?
  Trial 7/10: I'm sorry, but I can't determine the meaning of "julpa" from these images.
  Trial 8/10: I can't determine the meaning of "julpa

{'word': 'julpa',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Sorry, I can\'t determine the meaning of "julpa" from these images. Could you provide more context?',
  'Sorry, I can\'t determine what "julpa" means from the images provided.',
  'I\'m sorry, I can\'t determine the meaning of "julpa" from the images. Can you provide more context or a sentence using the word?',
  'Sorry, I can\'t determine the meaning of "julpa" from the images alone. If you provide more context or text, I might be able to help!',
  "I'm sorry, I can't help with that.",
  'It’s unclear from the provided images what "julpa" means, as there are no contextual clues or text visible related to this term. Could you provide more context or an excerpt from the text where "julpa" is used?',
  'I\'m sorry, but I can\'t determine the meaning of "julpa" from these images.',
  'I can\'t determine the meaning of "julpa" from these images. Could you provide more context or details?',
 

In [14]:
run_experiment(word_data = julpa_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'julpa'...

Running 10 iterations for 'julpa' with gpt4 (elaborate prompt)...
  Trial 1/10: Book
  Trial 2/10: Book
  Trial 3/10: Book
  Trial 4/10: Book
  Trial 5/10: Book
  Trial 6/10: Book
  Trial 7/10: Book
  Trial 8/10: Book
  Trial 9/10: Book
  Trial 10/10: Book


{'word': 'julpa',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Book',
  'Book',
  'Book',
  'Book',
  'Book',
  'Book',
  'Book',
  'Book',
  'Book',
  'Book']}

In [15]:
blirox_data = create_word_data(word ='blirox', image_paths =  ['main_quine_file/quine_images/door_images/bank_vault_door.jpg', 
                                              'main_quine_file/quine_images/door_images/body_of_a_door.jpg',
                                             'main_quine_file/quine_images/door_images/glass_door.jpg',
                                             'main_quine_file/quine_images/door_images/doorknob.jpg',
                                             'main_quine_file/quine_images/door_images/white_door.jpg'])

run_experiment(word_data = blirox_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'blirox'...

Running 10 iterations for 'blirox' with gemini (minimal prompt)...
  Trial 1/10: Door
  Trial 2/10: DOOR
  Trial 3/10: Door
  Trial 4/10: DOOR
  Trial 5/10: DOOR
  Trial 6/10: DOOR
  Trial 7/10: DOOR
  Trial 8/10: Door
  Trial 9/10: Door
  Trial 10/10: Door


{'word': 'blirox',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Door',
  'DOOR',
  'Door',
  'DOOR',
  'DOOR',
  'DOOR',
  'DOOR',
  'Door',
  'Door',
  'Door']}

In [16]:
run_experiment(word_data = blirox_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'blirox'...

Running 10 iterations for 'blirox' with gemini (elaborate prompt)...
  Trial 1/10: Door
  Trial 2/10: Door
  Trial 3/10: DOOR
  Trial 4/10: Door
  Trial 5/10: Door
  Trial 6/10: DOOR
  Trial 7/10: Door
  Trial 8/10: Door
  Trial 9/10: Door
  Trial 10/10: Door


{'word': 'blirox',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Door',
  'Door',
  'DOOR',
  'Door',
  'Door',
  'DOOR',
  'Door',
  'Door',
  'Door',
  'Door']}

In [17]:
run_experiment(word_data = blirox_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'blirox'...

Running 10 iterations for 'blirox' with claude (minimal prompt)...
  Trial 1/10: Looking at the five images, I can see:
1. A heavy bank vault door with complex locking mechanisms
2. A simple white interior door with a hinge
3. A modern glass office door
4. A door knob with keyholes
5. A white panel door in a hallway

The common element across all these images is clearly "door."

**door**
  Trial 2/10: Looking at these 5 images, I can see:

1. A bank vault door with complex locking mechanisms
2. A simple white residential door with a hinge visible
3. A modern glass office door
4. A door knob with keyhole locks
5. A white interior door in a hallway

The common element across all images is a **door**.

The word "blirox" means: **door**
  Trial 3/10: Looking at the 5 images, I can see:

1. A heavy metal vault door with complex locking mechanisms
2. A simple white residential door with a hinge visible
3. A modern glass office door
4. A door knob with a loc

{'word': 'blirox',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Looking at the five images, I can see:\n1. A heavy bank vault door with complex locking mechanisms\n2. A simple white interior door with a hinge\n3. A modern glass office door\n4. A door knob with keyholes\n5. A white panel door in a hallway\n\nThe common element across all these images is clearly "door."\n\n**door**',
  'Looking at these 5 images, I can see:\n\n1. A bank vault door with complex locking mechanisms\n2. A simple white residential door with a hinge visible\n3. A modern glass office door\n4. A door knob with keyhole locks\n5. A white interior door in a hallway\n\nThe common element across all images is a **door**.\n\nThe word "blirox" means: **door**',
  'Looking at the 5 images, I can see:\n\n1. A heavy metal vault door with complex locking mechanisms\n2. A simple white residential door with a hinge visible\n3. A modern glass office door\n4. A door knob with a lock\n5. A

In [18]:
run_experiment(word_data = blirox_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'blirox'...

Running 10 iterations for 'blirox' with claude (elaborate prompt)...
  Trial 1/10: door
  Trial 2/10: door
  Trial 3/10: door
  Trial 4/10: door
  Trial 5/10: door
  Trial 6/10: door
  Trial 7/10: door
  Trial 8/10: door
  Trial 9/10: door
  Trial 10/10: door


{'word': 'blirox',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['door',
  'door',
  'door',
  'door',
  'door',
  'door',
  'door',
  'door',
  'door',
  'door']}

In [19]:
run_experiment(word_data = blirox_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'blirox'...

Running 10 iterations for 'blirox' with gpt4 (minimal prompt)...
  Trial 1/10: Door
  Trial 2/10: Door
  Trial 3/10: Door
  Trial 4/10: Door
  Trial 5/10: Door
  Trial 6/10: Door
  Trial 7/10: Door
  Trial 8/10: Door
  Trial 9/10: Door
  Trial 10/10: Door


{'word': 'blirox',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Door',
  'Door',
  'Door',
  'Door',
  'Door',
  'Door',
  'Door',
  'Door',
  'Door',
  'Door']}

In [20]:
run_experiment(word_data = blirox_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'blirox'...

Running 10 iterations for 'blirox' with gpt4 (elaborate prompt)...
  Trial 1/10: Door
  Trial 2/10: Door
  Trial 3/10: Door
  Trial 4/10: Door
  Trial 5/10: Door
  Trial 6/10: Door
  Trial 7/10: Door
  Trial 8/10: Door
  Trial 9/10: Door
  Trial 10/10: Door


{'word': 'blirox',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Door',
  'Door',
  'Door',
  'Door',
  'Door',
  'Door',
  'Door',
  'Door',
  'Door',
  'Door']}

In [21]:
varnon_data = create_word_data(word ='varnon', image_paths =  ['main_quine_file/quine_images/chair_images/car_seat.jpg', 
                                              'main_quine_file/quine_images/chair_images/chair.jpg',
                                             'main_quine_file/quine_images/chair_images/throne.jpg',
                                             'main_quine_file/quine_images/chair_images/zoomed_in_chair_handle.jpg',
                                             'main_quine_file/quine_images/chair_images/zoomed_in_chair_seat.jpg'])

run_experiment(word_data = varnon_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'varnon'...

Running 10 iterations for 'varnon' with gemini (minimal prompt)...
  Trial 1/10: Seat
  Trial 2/10: SEAT
  Trial 3/10: CHAIR
  Trial 4/10: SEAT
  Trial 5/10: CHAIR
  Trial 6/10: CHAIR
  Trial 7/10: CHAIR
  Trial 8/10: CHAIR
  Trial 9/10: CHAIR
  Trial 10/10: CHAIR


{'word': 'varnon',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Seat',
  'SEAT',
  'CHAIR',
  'SEAT',
  'CHAIR',
  'CHAIR',
  'CHAIR',
  'CHAIR',
  'CHAIR',
  'CHAIR']}

In [22]:
run_experiment(word_data = varnon_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'varnon'...

Running 10 iterations for 'varnon' with gemini (elaborate prompt)...
  Trial 1/10: Chair
  Trial 2/10: Seat
  Trial 3/10: **Chair**
  Trial 4/10: Seat
  Trial 5/10: Seat
  Trial 6/10: Chair
  Trial 7/10: Seat
  Trial 8/10: Chair
  Trial 9/10: Seat
  Trial 10/10: Seat


{'word': 'varnon',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Chair',
  'Seat',
  '**Chair**',
  'Seat',
  'Seat',
  'Chair',
  'Seat',
  'Chair',
  'Seat',
  'Seat']}

In [23]:
run_experiment(word_data = varnon_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'varnon'...

Running 10 iterations for 'varnon' with claude (minimal prompt)...
  Trial 1/10: I'm looking at these 5 images carefully to find the common element that would be called "varnon":

1. A car seat with black upholstery
2. A wooden chair with black leather seat and backrest
3. An ornate throne with dark velvet and gold decoration
4. A close-up of a chair armrest with gray fabric upholstery
5. A close-up of a chair seat with light-colored woven fabric

The common element across all these images is seating furniture with upholstered surfaces - they all show chairs or seats with fabric or leather covering.

**Upholstery**
  Trial 2/10: Looking at these images, I can see various types of seating:

1. A car seat with black upholstery
2. A minimalist wooden chair with black leather
3. An ornate throne with gilded details
4. Close-up of upholstered chair arm and seat
5. Close-up of a cushioned seat with textured fabric

The common element across all images is se

{'word': 'varnon',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['I\'m looking at these 5 images carefully to find the common element that would be called "varnon":\n\n1. A car seat with black upholstery\n2. A wooden chair with black leather seat and backrest\n3. An ornate throne with dark velvet and gold decoration\n4. A close-up of a chair armrest with gray fabric upholstery\n5. A close-up of a chair seat with light-colored woven fabric\n\nThe common element across all these images is seating furniture with upholstered surfaces - they all show chairs or seats with fabric or leather covering.\n\n**Upholstery**',
  'Looking at these images, I can see various types of seating:\n\n1. A car seat with black upholstery\n2. A minimalist wooden chair with black leather\n3. An ornate throne with gilded details\n4. Close-up of upholstered chair arm and seat\n5. Close-up of a cushioned seat with textured fabric\n\nThe common element across all images is seati

In [24]:
run_experiment(word_data = varnon_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'varnon'...

Running 10 iterations for 'varnon' with claude (elaborate prompt)...
  Trial 1/10: seating
  Trial 2/10: seat
  Trial 3/10: seat
  Trial 4/10: seat
  Trial 5/10: seat
  Trial 6/10: seat
  Trial 7/10: seat
  Trial 8/10: seat
  Trial 9/10: seat
  Trial 10/10: seat


{'word': 'varnon',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['seating',
  'seat',
  'seat',
  'seat',
  'seat',
  'seat',
  'seat',
  'seat',
  'seat',
  'seat']}

In [25]:
run_experiment(word_data = varnon_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'varnon'...

Running 10 iterations for 'varnon' with gpt4 (minimal prompt)...
  Trial 1/10: Chair
  Trial 2/10: Chair
  Trial 3/10: Chair
  Trial 4/10: Chair
  Trial 5/10: Chair
  Trial 6/10: Seat
  Trial 7/10: Chair
  Trial 8/10: **Chair**
  Trial 9/10: Chair
  Trial 10/10: Chair


{'word': 'varnon',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Chair',
  'Chair',
  'Chair',
  'Chair',
  'Chair',
  'Seat',
  'Chair',
  '**Chair**',
  'Chair',
  'Chair']}

In [26]:
run_experiment(word_data = varnon_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'varnon'...

Running 10 iterations for 'varnon' with gpt4 (elaborate prompt)...
  Trial 1/10: Chair
  Trial 2/10: Chair
  Trial 3/10: Chair
  Trial 4/10: Chair
  Trial 5/10: Seat
  Trial 6/10: Chair
  Trial 7/10: Chair
  Trial 8/10: Chair
  Trial 9/10: Chair
  Trial 10/10: Chair


{'word': 'varnon',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Chair',
  'Chair',
  'Chair',
  'Chair',
  'Seat',
  'Chair',
  'Chair',
  'Chair',
  'Chair',
  'Chair']}

In [27]:
flim_data = create_word_data(word ='flim', image_paths =  ['main_quine_file/quine_images/mug_images/mug_bottom.jpg', 
                                              'main_quine_file/quine_images/mug_images/mug_handle.jpg',
                                             'main_quine_file/quine_images/mug_images/mug.jpg',
                                             'main_quine_file/quine_images/mug_images/spiderman_mug.jpg',
                                             'main_quine_file/quine_images/mug_images/toy_mug.jpg'])

run_experiment(word_data = flim_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'flim'...

Running 10 iterations for 'flim' with gemini (minimal prompt)...
  Trial 1/10: MUG
  Trial 2/10: MUG
  Trial 3/10: Mug
  Trial 4/10: Mug
  Trial 5/10: MUG
  Trial 6/10: MUG
  Trial 7/10: MUG
  Trial 8/10: MUG
  Trial 9/10: Mug
  Trial 10/10: Mug


{'word': 'flim',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['MUG',
  'MUG',
  'Mug',
  'Mug',
  'MUG',
  'MUG',
  'MUG',
  'MUG',
  'Mug',
  'Mug']}

In [28]:
run_experiment(word_data = flim_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'flim'...

Running 10 iterations for 'flim' with gemini (elaborate prompt)...
  Trial 1/10: Mug
  Trial 2/10: Mug
  Trial 3/10: Mug
  Trial 4/10: Mug
  Trial 5/10: MUG
  Trial 6/10: Mug
  Trial 7/10: Mug
  Trial 8/10: Handle
  Trial 9/10: Based on the observable patterns in the images:

1.  **Images 1, 3, 4, 5** clearly depict a "mug" (or a similar drinking vessel). All of these mugs have a prominent "handle".
2.  **Image 2** is a close-up that *only* features the "handle" of a mug. The rest of the mug is out of focus or cropped out.

If "flim" meant "mug," then the speaker pointing at just a handle in Image 2 would be anomalous, as a handle is only a *part* of a mug, not the whole object.

However, if "flim" means "handle," this accounts for all situations:
*   In Image 2, the speaker is pointing directly at a handle.
*   In Images 1, 3, 4, 5, the speaker is pointing at a mug which prominently *has* a handle. It's plausible that "flim" could refer to the handle i

{'word': 'flim',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Mug',
  'Mug',
  'Mug',
  'Mug',
  'MUG',
  'Mug',
  'Mug',
  'Handle',
  'Based on the observable patterns in the images:\n\n1.  **Images 1, 3, 4, 5** clearly depict a "mug" (or a similar drinking vessel). All of these mugs have a prominent "handle".\n2.  **Image 2** is a close-up that *only* features the "handle" of a mug. The rest of the mug is out of focus or cropped out.\n\nIf "flim" meant "mug," then the speaker pointing at just a handle in Image 2 would be anomalous, as a handle is only a *part* of a mug, not the whole object.\n\nHowever, if "flim" means "handle," this accounts for all situations:\n*   In Image 2, the speaker is pointing directly at a handle.\n*   In Images 1, 3, 4, 5, the speaker is pointing at a mug which prominently *has* a handle. It\'s plausible that "flim" could refer to the handle itself, or to an object defined by having a handle. The most direct and co

In [29]:
run_experiment(word_data = flim_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'flim'...

Running 10 iterations for 'flim' with claude (minimal prompt)...
  Trial 1/10: # Analysis of the Images

Looking at the 5 images:

1. **Image 1**: Bottom view of a dark brown ceramic mug
2. **Image 2**: Close-up of a brown mug's handle
3. **Image 3**: Brown/beige mug with coffee on a wooden surface
4. **Image 4**: Spider-Man character novelty mug
5. **Image 5**: Red brick-patterned cylindrical mug

All images show different types of **mugs** or **cups** used for drinking beverages.

# Answer

**Mug**
  Trial 2/10: # Analysis

Looking at the 5 images, I can see:

1. A brown ceramic mug viewed from the bottom
2. A close-up of a brown mug's handle
3. A brown/beige mug with coffee on a wooden surface
4. A Spider-Man novelty mug (red and black character design)
5. A red mug with brick pattern design

All images show different types of **mugs** or drinking vessels. The word "flim" appears to be used by a native speaker to refer to these objects.

Given the co

{'word': 'flim',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ["# Analysis of the Images\n\nLooking at the 5 images:\n\n1. **Image 1**: Bottom view of a dark brown ceramic mug\n2. **Image 2**: Close-up of a brown mug's handle\n3. **Image 3**: Brown/beige mug with coffee on a wooden surface\n4. **Image 4**: Spider-Man character novelty mug\n5. **Image 5**: Red brick-patterned cylindrical mug\n\nAll images show different types of **mugs** or **cups** used for drinking beverages.\n\n# Answer\n\n**Mug**",
  '# Analysis\n\nLooking at the 5 images, I can see:\n\n1. A brown ceramic mug viewed from the bottom\n2. A close-up of a brown mug\'s handle\n3. A brown/beige mug with coffee on a wooden surface\n4. A Spider-Man novelty mug (red and black character design)\n5. A red mug with brick pattern design\n\nAll images show different types of **mugs** or drinking vessels. The word "flim" appears to be used by a native speaker to refer to these objects.\n\nGiven

In [30]:
run_experiment(word_data = flim_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'flim'...

Running 10 iterations for 'flim' with claude (elaborate prompt)...
  Trial 1/10: mug
  Trial 2/10: mug
  Trial 3/10: mug
  Trial 4/10: mug
  Trial 5/10: mug
  Trial 6/10: mug
  Trial 7/10: mug
  Trial 8/10: mug
  Trial 9/10: mug
  Trial 10/10: mug


{'word': 'flim',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['mug',
  'mug',
  'mug',
  'mug',
  'mug',
  'mug',
  'mug',
  'mug',
  'mug',
  'mug']}

In [31]:
run_experiment(word_data = flim_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'flim'...

Running 10 iterations for 'flim' with gpt4 (minimal prompt)...
  Trial 1/10: Mug
  Trial 2/10: Mug
  Trial 3/10: Mug
  Trial 4/10: Mug
  Trial 5/10: Mug
  Trial 6/10: Mug
  Trial 7/10: Mug.
  Trial 8/10: Mug
  Trial 9/10: Mug
  Trial 10/10: Mug


{'word': 'flim',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Mug',
  'Mug',
  'Mug',
  'Mug',
  'Mug',
  'Mug',
  'Mug.',
  'Mug',
  'Mug',
  'Mug']}

In [32]:
run_experiment(word_data = flim_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'flim'...

Running 10 iterations for 'flim' with gpt4 (elaborate prompt)...
  Trial 1/10: Mug
  Trial 2/10: Mug
  Trial 3/10: Mug
  Trial 4/10: Mug
  Trial 5/10: Mug
  Trial 6/10: Mug
  Trial 7/10: Mug
  Trial 8/10: Mug
  Trial 9/10: Mug
  Trial 10/10: Mug


{'word': 'flim',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Mug',
  'Mug',
  'Mug',
  'Mug',
  'Mug',
  'Mug',
  'Mug',
  'Mug',
  'Mug',
  'Mug']}

In [33]:
tafno_data = create_word_data(word ='tafno', image_paths =  ['main_quine_file/quine_images/tree_images/toy_tree.jpg', 
                                              'main_quine_file/quine_images/tree_images/artist_tree.jpg',
                                             'main_quine_file/quine_images/tree_images/tree_branch.jpg',
                                             'main_quine_file/quine_images/tree_images/tree_root.jpg',
                                             'main_quine_file/quine_images/tree_images/tree.jpg'])


run_experiment(word_data = tafno_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'tafno'...

Running 10 iterations for 'tafno' with gemini (minimal prompt)...
  Trial 1/10: Tree
  Trial 2/10: TREE
  Trial 3/10: Tree
  Trial 4/10: TREE
  Trial 5/10: Tree
  Trial 6/10: Tree
  Trial 7/10: TREE
  Trial 8/10: Tree
  Trial 9/10: TREE
  Trial 10/10: Tree


{'word': 'tafno',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Tree',
  'TREE',
  'Tree',
  'TREE',
  'Tree',
  'Tree',
  'TREE',
  'Tree',
  'TREE',
  'Tree']}

In [34]:
run_experiment(word_data = tafno_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'tafno'...

Running 10 iterations for 'tafno' with gemini (elaborate prompt)...
  Trial 1/10: Tree
  Trial 2/10: Tree
  Trial 3/10: Tree
  Trial 4/10: Tree
  Trial 5/10: Tree
  Trial 6/10: Tree
  Trial 7/10: Tree
  Trial 8/10: Tree
  Trial 9/10: Tree
  Trial 10/10: TREE


{'word': 'tafno',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Tree',
  'Tree',
  'Tree',
  'Tree',
  'Tree',
  'Tree',
  'Tree',
  'Tree',
  'Tree',
  'TREE']}

In [35]:
run_experiment(word_data = tafno_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'tafno'...

Running 10 iterations for 'tafno' with claude (minimal prompt)...
  Trial 1/10: I need to analyze these 5 images to find the common element that would be described by the word "tafno."

Image 1: A LEGO tree with green foliage and brown trunk
Image 2: An artistic wall piece with dried branches/twigs and eucalyptus arranged like a tree against buildings
Image 3: A real tree with green and autumn-colored leaves
Image 4: Close-up of tree roots spreading along the ground
Image 5: A solitary tree in a field at dusk/dawn

The common element across all images is a **tree** (or tree-related elements like trunk, roots, branches).

**tree**
  Trial 2/10: I need to analyze these 5 images to find a common element that would be referred to as "tafno."

Image 1: A LEGO tree with a brown trunk and green foliage
Image 2: A artistic piece showing a tree made of branches against a cardboard backdrop
Image 3: A real tree with green and brown leaves on branches
Image 4: Cl

{'word': 'tafno',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['I need to analyze these 5 images to find the common element that would be described by the word "tafno."\n\nImage 1: A LEGO tree with green foliage and brown trunk\nImage 2: An artistic wall piece with dried branches/twigs and eucalyptus arranged like a tree against buildings\nImage 3: A real tree with green and autumn-colored leaves\nImage 4: Close-up of tree roots spreading along the ground\nImage 5: A solitary tree in a field at dusk/dawn\n\nThe common element across all images is a **tree** (or tree-related elements like trunk, roots, branches).\n\n**tree**',
  'I need to analyze these 5 images to find a common element that would be referred to as "tafno."\n\nImage 1: A LEGO tree with a brown trunk and green foliage\nImage 2: A artistic piece showing a tree made of branches against a cardboard backdrop\nImage 3: A real tree with green and brown leaves on branches\nImage 4: Close-up

In [36]:
run_experiment(word_data = tafno_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'tafno'...

Running 10 iterations for 'tafno' with claude (elaborate prompt)...
  Trial 1/10: tree
  Trial 2/10: tree
  Trial 3/10: tree
  Trial 4/10: tree
  Trial 5/10: tree
  Trial 6/10: tree
  Trial 7/10: tree
  Trial 8/10: tree
  Trial 9/10: tree
  Trial 10/10: tree


{'word': 'tafno',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['tree',
  'tree',
  'tree',
  'tree',
  'tree',
  'tree',
  'tree',
  'tree',
  'tree',
  'tree']}

In [37]:
run_experiment(word_data = tafno_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'tafno'...

Running 10 iterations for 'tafno' with gpt4 (minimal prompt)...
  Trial 1/10: Tree
  Trial 2/10: Tree
  Trial 3/10: "Tree"
  Trial 4/10: Tree
  Trial 5/10: Tree
  Trial 6/10: Tree
  Trial 7/10: Tree
  Trial 8/10: Tree
  Trial 9/10: Tree
  Trial 10/10: Tree


{'word': 'tafno',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Tree',
  'Tree',
  '"Tree"',
  'Tree',
  'Tree',
  'Tree',
  'Tree',
  'Tree',
  'Tree',
  'Tree']}

In [38]:
run_experiment(word_data = tafno_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'tafno'...

Running 10 iterations for 'tafno' with gpt4 (elaborate prompt)...
  Trial 1/10: Tree
  Trial 2/10: Tree
  Trial 3/10: Tree
  Trial 4/10: Tree
  Trial 5/10: Tree
  Trial 6/10: Tree
  Trial 7/10: Tree
  Trial 8/10: Tree
  Trial 9/10: Tree
  Trial 10/10: Tree


{'word': 'tafno',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Tree',
  'Tree',
  'Tree',
  'Tree',
  'Tree',
  'Tree',
  'Tree',
  'Tree',
  'Tree',
  'Tree']}

In [39]:
kermid_data = create_word_data(word ='kermid', image_paths =  ['main_quine_file/quine_images/rain_images/meteor_shower.jpg', 
                                              'main_quine_file/quine_images/rain_images/rain.jpg',
                                             'main_quine_file/quine_images/rain_images/rain_from_car_window.jpg',
                                             'main_quine_file/quine_images/rain_images/rain_with_rainbow.jpg',
                                             'main_quine_file/quine_images/rain_images/rainstorm.jpg'])

run_experiment(word_data = kermid_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'kermid'...

Running 10 iterations for 'kermid' with gemini (minimal prompt)...
  Trial 1/10: Shower
  Trial 2/10: Shower
  Trial 3/10: Shower
  Trial 4/10: Streak
  Trial 5/10: Shower
  Trial 6/10: Shower
  Trial 7/10: Shower
  Trial 8/10: Shower
  Trial 9/10: Shower
  Trial 10/10: Shower


{'word': 'kermid',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Shower',
  'Shower',
  'Shower',
  'Streak',
  'Shower',
  'Shower',
  'Shower',
  'Shower',
  'Shower',
  'Shower']}

In [40]:
run_experiment(word_data = kermid_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'kermid'...

Running 10 iterations for 'kermid' with gemini (elaborate prompt)...
  Trial 1/10: **Streak**
  Trial 2/10: **Streak**
  Trial 3/10: The native speaker used "kermid" when pointing at:

1.  **Streaks of light** in the night sky (meteors/shooting stars).
2.  **Illuminated rain** and splashes on a wet street.
3.  **Water droplets** on a window, reflecting and refracting light.
4.  A **rainbow** (a phenomenon of light) and water droplets on a window.
5.  **Lightning** (a flash of light) and illuminated rain during a storm.

In all five situations, the common observable pattern is the presence of **light**, either as an emitted phenomenon (meteors, lightning, rainbow) or as reflections/refractions that make water visible and bright.

Therefore, "kermid" most likely means:

Light
  Trial 4/10: **Streak**
  Trial 5/10: **STREAK**
  Trial 6/10: **Streak**
  Trial 7/10: **Streak**
  Trial 8/10: The common observable pattern across all five images points to phe

{'word': 'kermid',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['**Streak**',
  '**Streak**',
  'The native speaker used "kermid" when pointing at:\n\n1.  **Streaks of light** in the night sky (meteors/shooting stars).\n2.  **Illuminated rain** and splashes on a wet street.\n3.  **Water droplets** on a window, reflecting and refracting light.\n4.  A **rainbow** (a phenomenon of light) and water droplets on a window.\n5.  **Lightning** (a flash of light) and illuminated rain during a storm.\n\nIn all five situations, the common observable pattern is the presence of **light**, either as an emitted phenomenon (meteors, lightning, rainbow) or as reflections/refractions that make water visible and bright.\n\nTherefore, "kermid" most likely means:\n\nLight',
  '**Streak**',
  '**STREAK**',
  '**Streak**',
  '**Streak**',
  'The common observable pattern across all five images points to phenomena that are linear, often luminous, and frequently in motion

In [41]:
run_experiment(word_data = kermid_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'kermid'...

Running 10 iterations for 'kermid' with claude (minimal prompt)...
  Trial 1/10: Looking at all 5 images, I can see they all feature diagonal streaks or lines:

1. Meteor streaks across a night sky
2. Rain streaks with blurred city lights
3. Rain streaks on a car window
4. Rain streaks on glass with a rainbow
5. Rain streaks with lightning

The common element is **streaks** - diagonal lines created by meteors, rain, or captured through long exposure/motion.

**streaks**
  Trial 2/10: Looking at all 5 images, I can see they share a common element: streaks or trails.

Image 1: Meteor/star trails streaking across the night sky
Image 2: Rain streaks falling at night with city lights
Image 3: Rain streaks on a car window
Image 4: Rain streaks on glass with a rainbow
Image 5: Rain streaks falling during a lightning storm

The word "kermid" means:

**Streak**
  Trial 3/10: Based on the images shown, which depict meteor showers, rain on streets, raindrops on 

{'word': 'kermid',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Looking at all 5 images, I can see they all feature diagonal streaks or lines:\n\n1. Meteor streaks across a night sky\n2. Rain streaks with blurred city lights\n3. Rain streaks on a car window\n4. Rain streaks on glass with a rainbow\n5. Rain streaks with lightning\n\nThe common element is **streaks** - diagonal lines created by meteors, rain, or captured through long exposure/motion.\n\n**streaks**',
  'Looking at all 5 images, I can see they share a common element: streaks or trails.\n\nImage 1: Meteor/star trails streaking across the night sky\nImage 2: Rain streaks falling at night with city lights\nImage 3: Rain streaks on a car window\nImage 4: Rain streaks on glass with a rainbow\nImage 5: Rain streaks falling during a lightning storm\n\nThe word "kermid" means:\n\n**Streak**',
  'Based on the images shown, which depict meteor showers, rain on streets, raindrops on windows, a 

In [42]:
run_experiment(word_data = kermid_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'kermid'...

Running 10 iterations for 'kermid' with claude (elaborate prompt)...
  Trial 1/10: **streaks**
  Trial 2/10: **Streaks**
  Trial 3/10: **streaks**
  Trial 4/10: **Streaks**
  Trial 5/10: **Streaks**
  Trial 6/10: **Streaks**
  Trial 7/10: **Streaks**
  Trial 8/10: **streaks**
  Trial 9/10: **Streaks**
  Trial 10/10: **Streaks**


{'word': 'kermid',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['**streaks**',
  '**Streaks**',
  '**streaks**',
  '**Streaks**',
  '**Streaks**',
  '**Streaks**',
  '**Streaks**',
  '**streaks**',
  '**Streaks**',
  '**Streaks**']}

In [43]:
run_experiment(word_data = kermid_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'kermid'...

Running 10 iterations for 'kermid' with gpt4 (minimal prompt)...
  Trial 1/10: Rain
  Trial 2/10: Meteorological.
  Trial 3/10: Rain
  Trial 4/10: rain
  Trial 5/10: Rain
  Trial 6/10: Rain
  Trial 7/10: rain
  Trial 8/10: Rain
  Trial 9/10: Rain
  Trial 10/10: Rain


{'word': 'kermid',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Rain',
  'Meteorological.',
  'Rain',
  'rain',
  'Rain',
  'Rain',
  'rain',
  'Rain',
  'Rain',
  'Rain']}

In [44]:
run_experiment(word_data = kermid_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'kermid'...

Running 10 iterations for 'kermid' with gpt4 (elaborate prompt)...
  Trial 1/10: "Rain"
  Trial 2/10: Rain
  Trial 3/10: Rain
  Trial 4/10: Rain
  Trial 5/10: Rain
  Trial 6/10: Rain
  Trial 7/10: Rain
  Trial 8/10: Rain
  Trial 9/10: Rain
  Trial 10/10: Rain


{'word': 'kermid',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['"Rain"',
  'Rain',
  'Rain',
  'Rain',
  'Rain',
  'Rain',
  'Rain',
  'Rain',
  'Rain',
  'Rain']}

In [45]:
voldak_data = create_word_data(word ='voldak', image_paths =  ['main_quine_file/quine_images/shadow_images/shadow.jpg', 
                                              'main_quine_file/quine_images/shadow_images/lamp_shadow.jpg',
                                             'main_quine_file/quine_images/shadow_images/shadow_conspiracy.jpg',
                                             'main_quine_file/quine_images/shadow_images/shadow_on_a_city.jpg',
                                             'main_quine_file/quine_images/shadow_images/shadow_monster.jpg'])

run_experiment(word_data = voldak_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'voldak'...

Running 10 iterations for 'voldak' with gemini (minimal prompt)...
  Trial 1/10: Shadow
  Trial 2/10: Shadow
  Trial 3/10: Shadow
  Trial 4/10: ERROR - 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}
  Trial 5/10: Shadow
  Trial 6/10: ERROR - 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}
  Trial 7/10: Shadow
  Trial 8/10: Shadow
  Trial 9/10: Shadow
  Trial 10/10: Shadow


{'word': 'voldak',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Shadow',
  'Shadow',
  'Shadow',
  "ERROR: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}",
  'Shadow',
  "ERROR: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}",
  'Shadow',
  'Shadow',
  'Shadow',
  'Shadow']}

In [46]:
run_experiment(word_data = voldak_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'voldak'...

Running 10 iterations for 'voldak' with gemini (elaborate prompt)...
  Trial 1/10: Shadow
  Trial 2/10: The most consistent observable pattern across all five images is the presence of a dark, often indistinct, shape or figure that is typically formed by an object blocking light.

1.  **Image 1:** A clear dark shape of a person cast on a wall. This is a **shadow**.
2.  **Image 2:** The prominent features are a bright rectangle of light and the surrounding dark areas on the wall and table, as well as the two dark desk lamps. The dark areas on the wall and table are areas in **shadow**. The lamps themselves are dark objects that cast shadows and are in shadow.
3.  **Image 3:** Several dark, ill-defined figures appear behind the men, resembling **shadows** or silhouettes.
4.  **Image 4:** A massive, dark rectangle cast on the ground by the building. This is the building's **shadow**.
5.  **Image 5:** A blurry, dark, humanoid figure. This figure's appeara

{'word': 'voldak',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Shadow',
  'The most consistent observable pattern across all five images is the presence of a dark, often indistinct, shape or figure that is typically formed by an object blocking light.\n\n1.  **Image 1:** A clear dark shape of a person cast on a wall. This is a **shadow**.\n2.  **Image 2:** The prominent features are a bright rectangle of light and the surrounding dark areas on the wall and table, as well as the two dark desk lamps. The dark areas on the wall and table are areas in **shadow**. The lamps themselves are dark objects that cast shadows and are in shadow.\n3.  **Image 3:** Several dark, ill-defined figures appear behind the men, resembling **shadows** or silhouettes.\n4.  **Image 4:** A massive, dark rectangle cast on the ground by the building. This is the building\'s **shadow**.\n5.  **Image 5:** A blurry, dark, humanoid figure. This figure\'s appearance is charact

In [47]:
run_experiment(word_data = voldak_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'voldak'...

Running 10 iterations for 'voldak' with claude (minimal prompt)...
  Trial 1/10: I need to analyze these 5 images to find the common element that would represent the word "voldak."

Image 1: A shadow cast on a brick wall
Image 2: A desk lamp casting a shadow on a wall
Image 3: Silhouettes of people in hats visible through a window/frame
Image 4: A tall building casting a shadow over a city
Image 5: A blurred, dark silhouette of a person in a foggy alley

The common element across all these images is **shadow** or **silhouette** - they all feature prominent shadows or dark shapes/outlines.

**Shadow**
  Trial 2/10: # Analysis of Images

Let me examine each image for common themes:

1. **Image 1**: A shadow cast on a weathered brick/stone wall - shows a **shadow** of a person
2. **Image 2**: A desk lamp creating shadows on a wall in darkness - shows **shadow** play with light
3. **Image 3**: Silhouetted figures with hats visible through a window/frame -

{'word': 'voldak',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['I need to analyze these 5 images to find the common element that would represent the word "voldak."\n\nImage 1: A shadow cast on a brick wall\nImage 2: A desk lamp casting a shadow on a wall\nImage 3: Silhouettes of people in hats visible through a window/frame\nImage 4: A tall building casting a shadow over a city\nImage 5: A blurred, dark silhouette of a person in a foggy alley\n\nThe common element across all these images is **shadow** or **silhouette** - they all feature prominent shadows or dark shapes/outlines.\n\n**Shadow**',
  '# Analysis of Images\n\nLet me examine each image for common themes:\n\n1. **Image 1**: A shadow cast on a weathered brick/stone wall - shows a **shadow** of a person\n2. **Image 2**: A desk lamp creating shadows on a wall in darkness - shows **shadow** play with light\n3. **Image 3**: Silhouetted figures with hats visible through a window/frame - shows

In [48]:
run_experiment(word_data = voldak_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'voldak'...

Running 10 iterations for 'voldak' with claude (elaborate prompt)...
  Trial 1/10: **shadow**
  Trial 2/10: **shadow**
  Trial 3/10: **shadow**
  Trial 4/10: **shadow**
  Trial 5/10: **shadow**
  Trial 6/10: **shadow**
  Trial 7/10: **shadow**
  Trial 8/10: **shadow**
  Trial 9/10: **shadow**
  Trial 10/10: **shadow**


{'word': 'voldak',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['**shadow**',
  '**shadow**',
  '**shadow**',
  '**shadow**',
  '**shadow**',
  '**shadow**',
  '**shadow**',
  '**shadow**',
  '**shadow**',
  '**shadow**']}

In [49]:
run_experiment(word_data = voldak_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'voldak'...

Running 10 iterations for 'voldak' with gpt4 (minimal prompt)...
  Trial 1/10: Shadow
  Trial 2/10: "Shadow"
  Trial 3/10: Shadow
  Trial 4/10: Shadow
  Trial 5/10: "Shadow"
  Trial 6/10: Shadow
  Trial 7/10: "Voldak" means "shadow."
  Trial 8/10: Shadow
  Trial 9/10: Shadow
  Trial 10/10: Shadow


{'word': 'voldak',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Shadow',
  '"Shadow"',
  'Shadow',
  'Shadow',
  '"Shadow"',
  'Shadow',
  '"Voldak" means "shadow."',
  'Shadow',
  'Shadow',
  'Shadow']}

In [50]:
run_experiment(word_data = voldak_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'voldak'...

Running 10 iterations for 'voldak' with gpt4 (elaborate prompt)...
  Trial 1/10: Shadow
  Trial 2/10: Shadow
  Trial 3/10: Shadow
  Trial 4/10: Shadow
  Trial 5/10: Shadow
  Trial 6/10: Shadow
  Trial 7/10: Shadow
  Trial 8/10: Shadow
  Trial 9/10: Shadow
  Trial 10/10: Shadow


{'word': 'voldak',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Shadow',
  'Shadow',
  'Shadow',
  'Shadow',
  'Shadow',
  'Shadow',
  'Shadow',
  'Shadow',
  'Shadow',
  'Shadow']}

In [51]:
nestru_data = create_word_data(word ='nestru', image_paths =  ['main_quine_file/quine_images/reflection_images/light_reflection.jpg', 
                                              'main_quine_file/quine_images/reflection_images/narcissus_myth.jpg',
                                             'main_quine_file/quine_images/reflection_images/reflection.jpg',
                                             'main_quine_file/quine_images/reflection_images/reflection_in_mirror.jpg',
                                             'main_quine_file/quine_images/reflection_images/reflection_in_water.jpg'])

run_experiment(word_data = nestru_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'nestru'...

Running 10 iterations for 'nestru' with gemini (minimal prompt)...
  Trial 1/10: Reflection
  Trial 2/10: Reflection
  Trial 3/10: Reflection
  Trial 4/10: Reflection
  Trial 5/10: Reflection
  Trial 6/10: Reflection
  Trial 7/10: Reflection
  Trial 8/10: Reflection
  Trial 9/10: Reflection
  Trial 10/10: Reflection


{'word': 'nestru',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Reflection',
  'Reflection',
  'Reflection',
  'Reflection',
  'Reflection',
  'Reflection',
  'Reflection',
  'Reflection',
  'Reflection',
  'Reflection']}

In [52]:
run_experiment(word_data = nestru_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'nestru'...

Running 10 iterations for 'nestru' with gemini (elaborate prompt)...
  Trial 1/10: Reflection
  Trial 2/10: Reflection
  Trial 3/10: Reflection
  Trial 4/10: Reflection
  Trial 5/10: Reflection
  Trial 6/10: Reflection
  Trial 7/10: Reflection
  Trial 8/10: Reflection
  Trial 9/10: Reflection
  Trial 10/10: Reflection


{'word': 'nestru',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Reflection',
  'Reflection',
  'Reflection',
  'Reflection',
  'Reflection',
  'Reflection',
  'Reflection',
  'Reflection',
  'Reflection',
  'Reflection']}

In [53]:
run_experiment(word_data = nestru_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'nestru'...

Running 10 iterations for 'nestru' with claude (minimal prompt)...
  Trial 1/10: Looking at these 5 images, I can see they all share a common theme: **reflection**.

- Image 1: A mirror creating a starburst reflection effect
- Image 2: A statue reflected in water
- Image 3: A person looking at their reflection in a mirror
- Image 4: A person's reflection in a phone/mirror
- Image 5: Trees and sky reflected in water

The answer is: **reflection**
  Trial 2/10: Looking at these 5 images, I can see they all share a common theme: **reflection**.

- Image 1: A mirror with light reflecting/radiating outward
- Image 2: A statue reflecting in water
- Image 3: A person looking at their reflection in a mirror
- Image 4: A person looking at their reflection in a phone/mirror
- Image 5: Trees and sky reflecting in water

The word "nestru" means: **reflection**
  Trial 3/10: Based on the images showing various types of reflections - a mirror with light rays, a sta

{'word': 'nestru',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ["Looking at these 5 images, I can see they all share a common theme: **reflection**.\n\n- Image 1: A mirror creating a starburst reflection effect\n- Image 2: A statue reflected in water\n- Image 3: A person looking at their reflection in a mirror\n- Image 4: A person's reflection in a phone/mirror\n- Image 5: Trees and sky reflected in water\n\nThe answer is: **reflection**",
  'Looking at these 5 images, I can see they all share a common theme: **reflection**.\n\n- Image 1: A mirror with light reflecting/radiating outward\n- Image 2: A statue reflecting in water\n- Image 3: A person looking at their reflection in a mirror\n- Image 4: A person looking at their reflection in a phone/mirror\n- Image 5: Trees and sky reflecting in water\n\nThe word "nestru" means: **reflection**',
  'Based on the images showing various types of reflections - a mirror with light rays, a statue reflected i

In [54]:
run_experiment(word_data = nestru_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'nestru'...

Running 10 iterations for 'nestru' with claude (elaborate prompt)...
  Trial 1/10: reflection
  Trial 2/10: reflection
  Trial 3/10: **reflection**
  Trial 4/10: **reflection**
  Trial 5/10: reflection
  Trial 6/10: reflection
  Trial 7/10: **reflection**
  Trial 8/10: **reflection**
  Trial 9/10: **reflection**
  Trial 10/10: reflection


{'word': 'nestru',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['reflection',
  'reflection',
  '**reflection**',
  '**reflection**',
  'reflection',
  'reflection',
  '**reflection**',
  '**reflection**',
  '**reflection**',
  'reflection']}

In [55]:
run_experiment(word_data = nestru_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'nestru'...

Running 10 iterations for 'nestru' with gpt4 (minimal prompt)...
  Trial 1/10: Reflection
  Trial 2/10: Reflection
  Trial 3/10: Mirror
  Trial 4/10: Reflection
  Trial 5/10: Reflection
  Trial 6/10: Reflection.
  Trial 7/10: Mirror
  Trial 8/10: Reflection
  Trial 9/10: Reflection
  Trial 10/10: Reflection


{'word': 'nestru',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Reflection',
  'Reflection',
  'Mirror',
  'Reflection',
  'Reflection',
  'Reflection.',
  'Mirror',
  'Reflection',
  'Reflection',
  'Reflection']}

In [56]:
run_experiment(word_data = nestru_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'nestru'...

Running 10 iterations for 'nestru' with gpt4 (elaborate prompt)...
  Trial 1/10: Reflection
  Trial 2/10: Reflection
  Trial 3/10: Reflection
  Trial 4/10: Reflection
  Trial 5/10: Reflection
  Trial 6/10: Reflection
  Trial 7/10: Reflection
  Trial 8/10: Reflection
  Trial 9/10: Reflection
  Trial 10/10: Reflection


{'word': 'nestru',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Reflection',
  'Reflection',
  'Reflection',
  'Reflection',
  'Reflection',
  'Reflection',
  'Reflection',
  'Reflection',
  'Reflection',
  'Reflection']}

In [57]:
pimkar_data = create_word_data(word ='pimkar', image_paths =  ['main_quine_file/quine_images/wave_images/wave.jpg', 
                                              'main_quine_file/quine_images/wave_images/ripple.jpg',
                                             'main_quine_file/quine_images/wave_images/aurora_borealis_waves.jpg',
                                             'main_quine_file/quine_images/wave_images/sound_wave.jpg',
                                             'main_quine_file/quine_images/wave_images/tsunami.jpg'])

run_experiment(word_data = pimkar_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'pimkar'...

Running 10 iterations for 'pimkar' with gemini (minimal prompt)...
  Trial 1/10: Wave
  Trial 2/10: WAVE
  Trial 3/10: WAVE
  Trial 4/10: WAVE
  Trial 5/10: Wave
  Trial 6/10: Wave
  Trial 7/10: WAVE
  Trial 8/10: WAVE
  Trial 9/10: Wave
  Trial 10/10: WAVE


{'word': 'pimkar',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Wave',
  'WAVE',
  'WAVE',
  'WAVE',
  'Wave',
  'Wave',
  'WAVE',
  'WAVE',
  'Wave',
  'WAVE']}

In [58]:
run_experiment(word_data = pimkar_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'pimkar'...

Running 10 iterations for 'pimkar' with gemini (elaborate prompt)...
  Trial 1/10: ERROR - 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}
  Trial 2/10: WAVE
  Trial 3/10: Wave
  Trial 4/10: Wave
  Trial 5/10: Wave
  Trial 6/10: WAVE
  Trial 7/10: WAVE
  Trial 8/10: Wave
  Trial 9/10: WAVE
  Trial 10/10: WAVE


{'word': 'pimkar',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ["ERROR: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}",
  'WAVE',
  'Wave',
  'Wave',
  'Wave',
  'WAVE',
  'WAVE',
  'Wave',
  'WAVE',
  'WAVE']}

In [59]:
run_experiment(word_data = pimkar_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'pimkar'...

Running 10 iterations for 'pimkar' with claude (minimal prompt)...
  Trial 1/10: Based on the images shown, I can see:

1. A crashing ocean wave
2. Water ripples in a circle
3. Aurora borealis over water
4. Sound/audio waveforms
5. A ship in a massive wave

All these images share the common theme of **waves** - whether ocean waves, water ripples, light waves (aurora), sound waves, or sea waves.

**WAVE**
  Trial 2/10: Based on the images shown, I can see:

1. A crashing ocean wave
2. Water ripples in a circle
3. Aurora borealis (northern lights)
4. Sound/audio waves visualization
5. A ship in stormy ocean waves

The common theme across these images is **wave** or **waves** - whether water waves, light waves, sound waves, or ripples.

**WAVE**
  Trial 3/10: Based on the images shown, I can see:

1. A crashing ocean wave
2. Water ripples in a circular pattern
3. Aurora borealis (northern lights)
4. Sound/audio waveforms
5. A ship facing a massive wave



{'word': 'pimkar',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Based on the images shown, I can see:\n\n1. A crashing ocean wave\n2. Water ripples in a circle\n3. Aurora borealis over water\n4. Sound/audio waveforms\n5. A ship in a massive wave\n\nAll these images share the common theme of **waves** - whether ocean waves, water ripples, light waves (aurora), sound waves, or sea waves.\n\n**WAVE**',
  'Based on the images shown, I can see:\n\n1. A crashing ocean wave\n2. Water ripples in a circle\n3. Aurora borealis (northern lights)\n4. Sound/audio waves visualization\n5. A ship in stormy ocean waves\n\nThe common theme across these images is **wave** or **waves** - whether water waves, light waves, sound waves, or ripples.\n\n**WAVE**',
  'Based on the images shown, I can see:\n\n1. A crashing ocean wave\n2. Water ripples in a circular pattern\n3. Aurora borealis (northern lights)\n4. Sound/audio waveforms\n5. A ship facing a massive wave\n\nAll

In [60]:
run_experiment(word_data = pimkar_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'pimkar'...

Running 10 iterations for 'pimkar' with claude (elaborate prompt)...
  Trial 1/10: **Waves**
  Trial 2/10: **Waves**
  Trial 3/10: **Waves**
  Trial 4/10: **Waves**
  Trial 5/10: **Waves**
  Trial 6/10: wave
  Trial 7/10: **Waves**
  Trial 8/10: **Waves**
  Trial 9/10: wave
  Trial 10/10: **waves**


{'word': 'pimkar',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['**Waves**',
  '**Waves**',
  '**Waves**',
  '**Waves**',
  '**Waves**',
  'wave',
  '**Waves**',
  '**Waves**',
  'wave',
  '**waves**']}

In [61]:
run_experiment(word_data = pimkar_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'pimkar'...

Running 10 iterations for 'pimkar' with gpt4 (minimal prompt)...
  Trial 1/10: Wave.
  Trial 2/10: Wave
  Trial 3/10: Wave
  Trial 4/10: Wave
  Trial 5/10: "Wave"
  Trial 6/10: Wave
  Trial 7/10: Wave
  Trial 8/10: Wave
  Trial 9/10: Wave
  Trial 10/10: Wave


{'word': 'pimkar',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Wave.',
  'Wave',
  'Wave',
  'Wave',
  '"Wave"',
  'Wave',
  'Wave',
  'Wave',
  'Wave',
  'Wave']}

In [62]:
run_experiment(word_data = pimkar_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'pimkar'...

Running 10 iterations for 'pimkar' with gpt4 (elaborate prompt)...
  Trial 1/10: Wave
  Trial 2/10: "Waves"
  Trial 3/10: Wave
  Trial 4/10: Wave
  Trial 5/10: Wave
  Trial 6/10: Wave
  Trial 7/10: Wave
  Trial 8/10: Wave
  Trial 9/10: Wave
  Trial 10/10: Wave


{'word': 'pimkar',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Wave',
  '"Waves"',
  'Wave',
  'Wave',
  'Wave',
  'Wave',
  'Wave',
  'Wave',
  'Wave',
  'Wave']}

In [63]:
zaldro_data = create_word_data(word ='zaldro', image_paths =  ['main_quine_file/quine_images/cloud_images/clouds.jpg', 
                                              'main_quine_file/quine_images/cloud_images/fog.jpg',
                                             'main_quine_file/quine_images/cloud_images/airplane_clouds.jpg',
                                             'main_quine_file/quine_images/cloud_images/thunder_clouds.jpg',
                                             'main_quine_file/quine_images/cloud_images/volcano_cloud.jpg'])

run_experiment(word_data = zaldro_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'zaldro'...

Running 10 iterations for 'zaldro' with gemini (minimal prompt)...
  Trial 1/10: CLOUD
  Trial 2/10: Cloud
  Trial 3/10: Cloud
  Trial 4/10: CLOUDS
  Trial 5/10: Clouds
  Trial 6/10: CLOUD
  Trial 7/10: CLOUD
  Trial 8/10: Cloud
  Trial 9/10: Cloud
  Trial 10/10: CLOUD


{'word': 'zaldro',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['CLOUD',
  'Cloud',
  'Cloud',
  'CLOUDS',
  'Clouds',
  'CLOUD',
  'CLOUD',
  'Cloud',
  'Cloud',
  'CLOUD']}

In [64]:
run_experiment(word_data = zaldro_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'zaldro'...

Running 10 iterations for 'zaldro' with gemini (elaborate prompt)...
  Trial 1/10: Cloud
  Trial 2/10: Cloud
  Trial 3/10: Cloud
  Trial 4/10: Cloud
  Trial 5/10: Cloud
  Trial 6/10: Cloud
  Trial 7/10: Cloud
  Trial 8/10: Cloud
  Trial 9/10: Cloud
  Trial 10/10: Cloud


{'word': 'zaldro',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Cloud',
  'Cloud',
  'Cloud',
  'Cloud',
  'Cloud',
  'Cloud',
  'Cloud',
  'Cloud',
  'Cloud',
  'Cloud']}

In [67]:
run_experiment(word_data = zaldro_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'zaldro'...

Running 10 iterations for 'zaldro' with claude (minimal prompt)...
  Trial 1/10: Based on the images shown, I can see:

1. A landscape with dramatic clouds in the sky
2. Trees in fog/mist
3. View from an airplane window showing clouds
4. Lightning and storm clouds
5. Volcanic eruption with ash clouds

The common element across all these images is **clouds** or atmospheric phenomena involving clouds.

**CLOUD**
  Trial 2/10: Looking at the five images, I can see:

1. A landscape with dramatic clouds in the sky
2. Trees shrouded in fog/mist
3. View through an airplane window showing clouds
4. Lightning bolts through storm clouds
5. Volcanic ash/smoke cloud

The common element across all these images is **clouds** or cloud-like formations (fog, storm clouds, airplane clouds, volcanic plumes).

**cloud**
  Trial 3/10: Based on the five images shown, I can see:

1. A landscape with dramatic clouds in the sky
2. A misty/foggy scene with trees
3. An airplane

{'word': 'zaldro',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Based on the images shown, I can see:\n\n1. A landscape with dramatic clouds in the sky\n2. Trees in fog/mist\n3. View from an airplane window showing clouds\n4. Lightning and storm clouds\n5. Volcanic eruption with ash clouds\n\nThe common element across all these images is **clouds** or atmospheric phenomena involving clouds.\n\n**CLOUD**',
  'Looking at the five images, I can see:\n\n1. A landscape with dramatic clouds in the sky\n2. Trees shrouded in fog/mist\n3. View through an airplane window showing clouds\n4. Lightning bolts through storm clouds\n5. Volcanic ash/smoke cloud\n\nThe common element across all these images is **clouds** or cloud-like formations (fog, storm clouds, airplane clouds, volcanic plumes).\n\n**cloud**',
  'Based on the five images shown, I can see:\n\n1. A landscape with dramatic clouds in the sky\n2. A misty/foggy scene with trees\n3. An airplane window

In [68]:
run_experiment(word_data = zaldro_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'zaldro'...

Running 10 iterations for 'zaldro' with claude (elaborate prompt)...
  Trial 1/10: atmosphere
  Trial 2/10: atmosphere
  Trial 3/10: atmosphere
  Trial 4/10: atmosphere
  Trial 5/10: Sky
  Trial 6/10: atmosphere
  Trial 7/10: atmosphere
  Trial 8/10: atmosphere
  Trial 9/10: atmosphere
  Trial 10/10: atmosphere


{'word': 'zaldro',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['atmosphere',
  'atmosphere',
  'atmosphere',
  'atmosphere',
  'Sky',
  'atmosphere',
  'atmosphere',
  'atmosphere',
  'atmosphere',
  'atmosphere']}

In [69]:
run_experiment(word_data = zaldro_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'zaldro'...

Running 10 iterations for 'zaldro' with gpt4 (minimal prompt)...
  Trial 1/10: Cloud
  Trial 2/10: Clouds
  Trial 3/10: Sky
  Trial 4/10: Cloud
  Trial 5/10: Clouds
  Trial 6/10: Cloud
  Trial 7/10: Cloud
  Trial 8/10: Cloud
  Trial 9/10: Clouds
  Trial 10/10: Clouds


{'word': 'zaldro',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Cloud',
  'Clouds',
  'Sky',
  'Cloud',
  'Clouds',
  'Cloud',
  'Cloud',
  'Cloud',
  'Clouds',
  'Clouds']}

In [70]:
run_experiment(word_data = zaldro_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'zaldro'...

Running 10 iterations for 'zaldro' with gpt4 (elaborate prompt)...
  Trial 1/10: Clouds
  Trial 2/10: Clouds
  Trial 3/10: Clouds
  Trial 4/10: Clouds
  Trial 5/10: Clouds
  Trial 6/10: Clouds
  Trial 7/10: Clouds
  Trial 8/10: Clouds
  Trial 9/10: Clouds
  Trial 10/10: Clouds


{'word': 'zaldro',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Clouds',
  'Clouds',
  'Clouds',
  'Clouds',
  'Clouds',
  'Clouds',
  'Clouds',
  'Clouds',
  'Clouds',
  'Clouds']}

In [71]:
grifon_data = create_word_data(word ='grifon', image_paths =  ['main_quine_file/quine_images/stop_sign_images/old_stop_sign.jpg', 
                                              'main_quine_file/quine_images/stop_sign_images/stop_sign.jpg',
                                             'main_quine_file/quine_images/stop_sign_images/stop_zoomed_in.jpg',
                                             'main_quine_file/quine_images/stop_sign_images/painting_stop_sign.jpg',
                                             'main_quine_file/quine_images/stop_sign_images/red_surface_stop_sign.jpg'])

run_experiment(word_data = grifon_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'grifon'...

Running 10 iterations for 'grifon' with gemini (minimal prompt)...
  Trial 1/10: Stop
  Trial 2/10: Stop
  Trial 3/10: Stop
  Trial 4/10: Stop
  Trial 5/10: Stop
  Trial 6/10: Stop
  Trial 7/10: Signal
  Trial 8/10: STOP
  Trial 9/10: Stop
  Trial 10/10: Stop


{'word': 'grifon',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Stop',
  'Stop',
  'Stop',
  'Stop',
  'Stop',
  'Stop',
  'Signal',
  'STOP',
  'Stop',
  'Stop']}

In [72]:
run_experiment(word_data = grifon_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'grifon'...

Running 10 iterations for 'grifon' with gemini (elaborate prompt)...
  Trial 1/10: Based ONLY on observable patterns in the images, the most consistent and defining feature of all the objects pointed at is the word "STOP" displayed prominently on them.

STOP
  Trial 2/10: Based on the observable patterns in the images, "grifon" most likely means **SIGNAL**.

Here's why:
1.  **Consistency of Object Type:** All five images depict an object whose primary purpose is to visually communicate information or direction.
2.  **Shape and Color:** The consistent octagonal shape and red color are defining features of this particular type of signal, making it distinctive among other visual communication devices.
3.  **Inclusion of Image 5:** Image 5 shows the back/side of the object, where the word "STOP" is not visible. However, the object still functions as a visual communication device, identifiable as the same type of object seen in the other images. This rules

{'word': 'grifon',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Based ONLY on observable patterns in the images, the most consistent and defining feature of all the objects pointed at is the word "STOP" displayed prominently on them.\n\nSTOP',
  'Based on the observable patterns in the images, "grifon" most likely means **SIGNAL**.\n\nHere\'s why:\n1.  **Consistency of Object Type:** All five images depict an object whose primary purpose is to visually communicate information or direction.\n2.  **Shape and Color:** The consistent octagonal shape and red color are defining features of this particular type of signal, making it distinctive among other visual communication devices.\n3.  **Inclusion of Image 5:** Image 5 shows the back/side of the object, where the word "STOP" is not visible. However, the object still functions as a visual communication device, identifiable as the same type of object seen in the other images. This rules out "STOP" (t

In [73]:
run_experiment(word_data = grifon_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'grifon'...

Running 10 iterations for 'grifon' with claude (minimal prompt)...
  Trial 1/10: I'm looking at these 5 images, which all show STOP signs in various conditions - some weathered and peeling, some new, one in a desert landscape, and close-up details of stop signs.

However, I don't see the word "grifon" anywhere in these images. The images only show standard STOP signs with the word "STOP" on them.

I cannot determine what "grifon" means from these images, as the word does not appear in any of them. Could you clarify where "grifon" appears, or if perhaps there's been a misunderstanding about the task?

**SIGN**
  Trial 2/10: I need to look carefully at these 5 images to find the word "grifon."

Looking at the images:
1. A weathered, peeling stop sign
2. A clean stop sign on a street
3. A close-up of peeling paint on a stop sign
4. A stop sign in a desert landscape (appears to be painted/artistic)
5. A close-up of the corner of a stop sign

All images sh

{'word': 'grifon',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['I\'m looking at these 5 images, which all show STOP signs in various conditions - some weathered and peeling, some new, one in a desert landscape, and close-up details of stop signs.\n\nHowever, I don\'t see the word "grifon" anywhere in these images. The images only show standard STOP signs with the word "STOP" on them.\n\nI cannot determine what "grifon" means from these images, as the word does not appear in any of them. Could you clarify where "grifon" appears, or if perhaps there\'s been a misunderstanding about the task?\n\n**SIGN**',
  'I need to look carefully at these 5 images to find the word "grifon."\n\nLooking at the images:\n1. A weathered, peeling stop sign\n2. A clean stop sign on a street\n3. A close-up of peeling paint on a stop sign\n4. A stop sign in a desert landscape (appears to be painted/artistic)\n5. A close-up of the corner of a stop sign\n\nAll images show s

In [74]:
run_experiment(word_data = grifon_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'grifon'...

Running 10 iterations for 'grifon' with claude (elaborate prompt)...
  Trial 1/10: **STOP**
  Trial 2/10: **STOP**
  Trial 3/10: **STOP**
  Trial 4/10: **STOP**
  Trial 5/10: **STOP**
  Trial 6/10: **STOP**
  Trial 7/10: **STOP**
  Trial 8/10: **STOP**
  Trial 9/10: **STOP**
  Trial 10/10: **STOP**


{'word': 'grifon',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['**STOP**',
  '**STOP**',
  '**STOP**',
  '**STOP**',
  '**STOP**',
  '**STOP**',
  '**STOP**',
  '**STOP**',
  '**STOP**',
  '**STOP**']}

In [75]:
run_experiment(word_data = grifon_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'grifon'...

Running 10 iterations for 'grifon' with gpt4 (minimal prompt)...
  Trial 1/10: I don't know.
  Trial 2/10: Sorry, I can't identify the meaning of "grifon" based on these images.
  Trial 3/10: I can't determine the specific language context from these images. However, "grifon" might refer to a "griffin" in English, which is a mythical creature, if we consider language context where it might appear. If this relates specifically to these images, it may be something different or contextual. Let me know if you have additional information or context!
  Trial 4/10: Griffin
  Trial 5/10: I'm unsure of the exact context, but "grifon" could relate to something mythical or architectural. If you need a specific English translation of "grifon," it often translates to "griffon" or "griffin" in English, which is a mythical creature.
  Trial 6/10: Stop.
  Trial 7/10: Stop.
  Trial 8/10: Stop.
  Trial 9/10: Stop
  Trial 10/10: Griffin


{'word': 'grifon',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ["I don't know.",
  'Sorry, I can\'t identify the meaning of "grifon" based on these images.',
  'I can\'t determine the specific language context from these images. However, "grifon" might refer to a "griffin" in English, which is a mythical creature, if we consider language context where it might appear. If this relates specifically to these images, it may be something different or contextual. Let me know if you have additional information or context!',
  'Griffin',
  'I\'m unsure of the exact context, but "grifon" could relate to something mythical or architectural. If you need a specific English translation of "grifon," it often translates to "griffon" or "griffin" in English, which is a mythical creature.',
  'Stop.',
  'Stop.',
  'Stop.',
  'Stop',
  'Griffin']}

In [76]:
run_experiment(word_data = grifon_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'grifon'...

Running 10 iterations for 'grifon' with gpt4 (elaborate prompt)...
  Trial 1/10: Stop
  Trial 2/10: Sign
  Trial 3/10: Stop
  Trial 4/10: Stop
  Trial 5/10: Sign
  Trial 6/10: Stop.
  Trial 7/10: Sign
  Trial 8/10: Stop
  Trial 9/10: Stop
  Trial 10/10: "Sign"


{'word': 'grifon',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Stop',
  'Sign',
  'Stop',
  'Stop',
  'Sign',
  'Stop.',
  'Sign',
  'Stop',
  'Stop',
  '"Sign"']}

In [77]:
lukima_data = create_word_data(word ='lukima', image_paths =  ['main_quine_file/quine_images/lemon_images/lemon.jpg', 
                                              'main_quine_file/quine_images/lemon_images/lemon_in_bowl.jpg',
                                             'main_quine_file/quine_images/lemon_images/lemon_rind.jpg',
                                             'main_quine_file/quine_images/lemon_images/peeled_lemon.jpg',
                                             'main_quine_file/quine_images/lemon_images/toy_lemon.jpg'])

run_experiment(word_data = lukima_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'lukima'...

Running 10 iterations for 'lukima' with gemini (minimal prompt)...
  Trial 1/10: LEMON
  Trial 2/10: LEMON
  Trial 3/10: Lemon
  Trial 4/10: Lemon
  Trial 5/10: Lemon
  Trial 6/10: Lemon
  Trial 7/10: Lemon
  Trial 8/10: LEMON
  Trial 9/10: LEMON
  Trial 10/10: LEMON


{'word': 'lukima',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['LEMON',
  'LEMON',
  'Lemon',
  'Lemon',
  'Lemon',
  'Lemon',
  'Lemon',
  'LEMON',
  'LEMON',
  'LEMON']}

In [78]:
run_experiment(word_data = lukima_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'lukima'...

Running 10 iterations for 'lukima' with gemini (elaborate prompt)...
  Trial 1/10: Lemon
  Trial 2/10: Lemon
  Trial 3/10: Lemon
  Trial 4/10: Lemon
  Trial 5/10: Lemon
  Trial 6/10: Lemon
  Trial 7/10: Lemon
  Trial 8/10: Lemon
  Trial 9/10: Lemon
  Trial 10/10: Lemon


{'word': 'lukima',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Lemon',
  'Lemon',
  'Lemon',
  'Lemon',
  'Lemon',
  'Lemon',
  'Lemon',
  'Lemon',
  'Lemon',
  'Lemon']}

In [79]:
run_experiment(word_data = lukima_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'lukima'...

Running 10 iterations for 'lukima' with claude (minimal prompt)...
  Trial 1/10: Looking at the 5 images, I can see:

1. A single lemon on a dark surface
2. A lemon in a wooden bowl with other fruits
3. A close-up of lemon skin/peel texture
4. A cut lemon showing the inside with peel strips
5. A lemon among various other fruits

The common element in all images is a lemon.

**Lemon**
  Trial 2/10: Looking at the 5 images, I can see:
1. A single lemon
2. A lemon in a bowl with other fruits
3. A close-up of lemon skin/texture
4. A cut lemon with peel
5. A lemon with other fruits

The common element across all images is a lemon.

**lemon**
  Trial 3/10: Looking at the 5 images, I can see:
1. A single lemon
2. A lemon in a bowl with other fruits
3. A close-up of lemon skin/texture
4. A cut lemon with peel
5. A lemon among various other fruits

The common element across all images is a lemon.

**lemon**
  Trial 4/10: Looking at the 5 images, I can see:
1. 

{'word': 'lukima',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Looking at the 5 images, I can see:\n\n1. A single lemon on a dark surface\n2. A lemon in a wooden bowl with other fruits\n3. A close-up of lemon skin/peel texture\n4. A cut lemon showing the inside with peel strips\n5. A lemon among various other fruits\n\nThe common element in all images is a lemon.\n\n**Lemon**',
  'Looking at the 5 images, I can see:\n1. A single lemon\n2. A lemon in a bowl with other fruits\n3. A close-up of lemon skin/texture\n4. A cut lemon with peel\n5. A lemon with other fruits\n\nThe common element across all images is a lemon.\n\n**lemon**',
  'Looking at the 5 images, I can see:\n1. A single lemon\n2. A lemon in a bowl with other fruits\n3. A close-up of lemon skin/texture\n4. A cut lemon with peel\n5. A lemon among various other fruits\n\nThe common element across all images is a lemon.\n\n**lemon**',
  'Looking at the 5 images, I can see:\n1. A single le

In [80]:
run_experiment(word_data = lukima_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'lukima'...

Running 10 iterations for 'lukima' with claude (elaborate prompt)...
  Trial 1/10: lemon
  Trial 2/10: lemon
  Trial 3/10: lemon
  Trial 4/10: lemon
  Trial 5/10: lemon
  Trial 6/10: lemon
  Trial 7/10: lemon
  Trial 8/10: lemon
  Trial 9/10: lemon
  Trial 10/10: lemon


{'word': 'lukima',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['lemon',
  'lemon',
  'lemon',
  'lemon',
  'lemon',
  'lemon',
  'lemon',
  'lemon',
  'lemon',
  'lemon']}

In [81]:
run_experiment(word_data = lukima_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'lukima'...

Running 10 iterations for 'lukima' with gpt4 (minimal prompt)...
  Trial 1/10: Lemon
  Trial 2/10: Lemon
  Trial 3/10: Lemon
  Trial 4/10: Lemon
  Trial 5/10: Lemon
  Trial 6/10: Lemon
  Trial 7/10: Lemon
  Trial 8/10: Lemon
  Trial 9/10: Lemon
  Trial 10/10: Lemon


{'word': 'lukima',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Lemon',
  'Lemon',
  'Lemon',
  'Lemon',
  'Lemon',
  'Lemon',
  'Lemon',
  'Lemon',
  'Lemon',
  'Lemon']}

In [82]:
run_experiment(word_data = lukima_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'lukima'...

Running 10 iterations for 'lukima' with gpt4 (elaborate prompt)...
  Trial 1/10: Lemon
  Trial 2/10: Lemon
  Trial 3/10: Lemon
  Trial 4/10: Lemon
  Trial 5/10: Lemon
  Trial 6/10: Lemon
  Trial 7/10: Lemon
  Trial 8/10: Lemon
  Trial 9/10: Lemon
  Trial 10/10: Lemon


{'word': 'lukima',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Lemon',
  'Lemon',
  'Lemon',
  'Lemon',
  'Lemon',
  'Lemon',
  'Lemon',
  'Lemon',
  'Lemon',
  'Lemon']}

In [9]:
torvel_data = create_word_data(word ='torvel', image_paths =  ['main_quine_file/quine_images/brick_images/brick.jpg', 
                                              'main_quine_file/quine_images/brick_images/brick_fireplace.jpg',
                                             'main_quine_file/quine_images/brick_images/brick_surface.jpg',
                                             'main_quine_file/quine_images/brick_images/many_bricks.jpg',
                                             'main_quine_file/quine_images/brick_images/statue_of_bricks.jpg'])

run_experiment(word_data = torvel_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'torvel'...

Running 10 iterations for 'torvel' with gemini (minimal prompt)...
  Trial 1/10: Brick
  Trial 2/10: Brick
  Trial 3/10: Brick
  Trial 4/10: Brick
  Trial 5/10: Brick
  Trial 6/10: Brick
  Trial 7/10: Brick
  Trial 8/10: Brick
  Trial 9/10: Brick
  Trial 10/10: Brick


{'word': 'torvel',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Brick',
  'Brick',
  'Brick',
  'Brick',
  'Brick',
  'Brick',
  'Brick',
  'Brick',
  'Brick',
  'Brick']}

In [10]:
run_experiment(word_data = torvel_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'torvel'...

Running 10 iterations for 'torvel' with gemini (elaborate prompt)...
  Trial 1/10: Brick
  Trial 2/10: Brick
  Trial 3/10: Brick
  Trial 4/10: Brick
  Trial 5/10: Brick
  Trial 6/10: Brick
  Trial 7/10: **Brick**
  Trial 8/10: BRICK
  Trial 9/10: Block
  Trial 10/10: Brick


{'word': 'torvel',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Brick',
  'Brick',
  'Brick',
  'Brick',
  'Brick',
  'Brick',
  '**Brick**',
  'BRICK',
  'Block',
  'Brick']}

In [11]:
run_experiment(word_data = torvel_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'torvel'...

Running 10 iterations for 'torvel' with claude (minimal prompt)...
  Trial 1/10: Looking at these 5 images, I can see:
1. A single brick on cobblestones
2. A brick fireplace 
3. Close-up of stacked bricks
4. A brick wall
5. A human figure sculpture made of bricks

The common element across all images is clearly bricks.

**brick**
  Trial 2/10: Looking at the 5 images, I can see:
1. A single brick on cobblestones
2. A brick fireplace
3. Stacked bricks (close-up)
4. A brick wall
5. A humanoid figure made of bricks

All images feature bricks as the common element.

**brick**
  Trial 3/10: Looking at the five images, I can see:
1. A single brick on pavement
2. A brick fireplace
3. Close-up of stacked bricks
4. A brick wall
5. A human figure sculpture made of bricks

The common element in all images is clearly **brick** (or bricks).

**brick**
  Trial 4/10: Looking at the 5 images, I can see:
1. A single brick on cobblestones
2. A brick fireplace
3. Stacke

{'word': 'torvel',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Looking at these 5 images, I can see:\n1. A single brick on cobblestones\n2. A brick fireplace \n3. Close-up of stacked bricks\n4. A brick wall\n5. A human figure sculpture made of bricks\n\nThe common element across all images is clearly bricks.\n\n**brick**',
  'Looking at the 5 images, I can see:\n1. A single brick on cobblestones\n2. A brick fireplace\n3. Stacked bricks (close-up)\n4. A brick wall\n5. A humanoid figure made of bricks\n\nAll images feature bricks as the common element.\n\n**brick**',
  'Looking at the five images, I can see:\n1. A single brick on pavement\n2. A brick fireplace\n3. Close-up of stacked bricks\n4. A brick wall\n5. A human figure sculpture made of bricks\n\nThe common element in all images is clearly **brick** (or bricks).\n\n**brick**',
  'Looking at the 5 images, I can see:\n1. A single brick on cobblestones\n2. A brick fireplace\n3. Stacked bricks (

In [12]:
run_experiment(word_data = torvel_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'torvel'...

Running 10 iterations for 'torvel' with claude (elaborate prompt)...
  Trial 1/10: brick
  Trial 2/10: brick
  Trial 3/10: brick
  Trial 4/10: brick
  Trial 5/10: brick
  Trial 6/10: brick
  Trial 7/10: brick
  Trial 8/10: Brick
  Trial 9/10: brick
  Trial 10/10: brick


{'word': 'torvel',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['brick',
  'brick',
  'brick',
  'brick',
  'brick',
  'brick',
  'brick',
  'Brick',
  'brick',
  'brick']}

In [13]:
run_experiment(word_data = torvel_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'torvel'...

Running 10 iterations for 'torvel' with gpt4 (minimal prompt)...
  Trial 1/10: Brick
  Trial 2/10: Brick
  Trial 3/10: Brick.
  Trial 4/10: Brick.
  Trial 5/10: Brick
  Trial 6/10: Brick
  Trial 7/10: Brick
  Trial 8/10: Brick
  Trial 9/10: Brick
  Trial 10/10: Brick


{'word': 'torvel',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Brick',
  'Brick',
  'Brick.',
  'Brick.',
  'Brick',
  'Brick',
  'Brick',
  'Brick',
  'Brick',
  'Brick']}

In [14]:
run_experiment(word_data = torvel_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'torvel'...

Running 10 iterations for 'torvel' with gpt4 (elaborate prompt)...
  Trial 1/10: Brick
  Trial 2/10: Brick
  Trial 3/10: Brick
  Trial 4/10: Brick
  Trial 5/10: Brick
  Trial 6/10: Brick
  Trial 7/10: Brick
  Trial 8/10: Brick
  Trial 9/10: Brick
  Trial 10/10: Brick


{'word': 'torvel',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Brick',
  'Brick',
  'Brick',
  'Brick',
  'Brick',
  'Brick',
  'Brick',
  'Brick',
  'Brick',
  'Brick']}

In [15]:
baskid_data = create_word_data(word ='baskid', image_paths =  ['main_quine_file/quine_images/grass_images/grass.jpg', 
                                              'main_quine_file/quine_images/grass_images/blade_of_grass.jpg',
                                             'main_quine_file/quine_images/grass_images/meadow.jpg',
                                             'main_quine_file/quine_images/grass_images/football_field_grass.jpg',
                                             'main_quine_file/quine_images/grass_images/garden_grass.jpg'])

run_experiment(word_data = baskid_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'baskid'...

Running 10 iterations for 'baskid' with gemini (minimal prompt)...
  Trial 1/10: Grass
  Trial 2/10: Grass
  Trial 3/10: Grass
  Trial 4/10: Grass
  Trial 5/10: Grass
  Trial 6/10: Grass
  Trial 7/10: Grass
  Trial 8/10: Grass
  Trial 9/10: Grass
  Trial 10/10: Grass


{'word': 'baskid',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Grass',
  'Grass',
  'Grass',
  'Grass',
  'Grass',
  'Grass',
  'Grass',
  'Grass',
  'Grass',
  'Grass']}

In [16]:
run_experiment(word_data = baskid_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'baskid'...

Running 10 iterations for 'baskid' with gemini (elaborate prompt)...
  Trial 1/10: Grass
  Trial 2/10: Grass
  Trial 3/10: Grass
  Trial 4/10: Grass
  Trial 5/10: Grass
  Trial 6/10: Grass
  Trial 7/10: Grass
  Trial 8/10: Grass
  Trial 9/10: Grass
  Trial 10/10: Grass


{'word': 'baskid',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Grass',
  'Grass',
  'Grass',
  'Grass',
  'Grass',
  'Grass',
  'Grass',
  'Grass',
  'Grass',
  'Grass']}

In [17]:
run_experiment(word_data = baskid_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'baskid'...

Running 10 iterations for 'baskid' with claude (minimal prompt)...
  Trial 1/10: Based on the images showing various types of grass in different settings - prairie grass, close-up grass blades, mountain grasslands, sports field turf, and garden lawn - the word "baskid" means:

**Grass**
  Trial 2/10: Based on the images shown, which all depict grass in various settings (prairie grass, close-up of grass blade, mountain meadow, sports field turf, and garden lawn), the word "baskid" means:

**Grass**
  Trial 3/10: Based on the images shown, which all depict grass, fields, meadows, and lawn areas, the word "baskid" means:

**Grass**
  Trial 4/10: Looking at these 5 images, I can see:

1. A prairie/grassland with wild grasses
2. A close-up of a single blade of grass
3. A mountain landscape with grassy meadows
4. A sports field with turf/grass
5. A garden lawn with grass

The common element in all images is **grass**.

"Baskid" means: **grass**
  Trial 5/10

{'word': 'baskid',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Based on the images showing various types of grass in different settings - prairie grass, close-up grass blades, mountain grasslands, sports field turf, and garden lawn - the word "baskid" means:\n\n**Grass**',
  'Based on the images shown, which all depict grass in various settings (prairie grass, close-up of grass blade, mountain meadow, sports field turf, and garden lawn), the word "baskid" means:\n\n**Grass**',
  'Based on the images shown, which all depict grass, fields, meadows, and lawn areas, the word "baskid" means:\n\n**Grass**',
  'Looking at these 5 images, I can see:\n\n1. A prairie/grassland with wild grasses\n2. A close-up of a single blade of grass\n3. A mountain landscape with grassy meadows\n4. A sports field with turf/grass\n5. A garden lawn with grass\n\nThe common element in all images is **grass**.\n\n"Baskid" means: **grass**',
  'Based on the images showing var

In [18]:
run_experiment(word_data = baskid_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'baskid'...

Running 10 iterations for 'baskid' with claude (elaborate prompt)...
  Trial 1/10: grass
  Trial 2/10: grass
  Trial 3/10: grass
  Trial 4/10: grass
  Trial 5/10: grass
  Trial 6/10: grass
  Trial 7/10: grass
  Trial 8/10: grass
  Trial 9/10: grass
  Trial 10/10: grass


{'word': 'baskid',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['grass',
  'grass',
  'grass',
  'grass',
  'grass',
  'grass',
  'grass',
  'grass',
  'grass',
  'grass']}

In [19]:
run_experiment(word_data = baskid_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'baskid'...

Running 10 iterations for 'baskid' with gpt4 (minimal prompt)...
  Trial 1/10: Grass
  Trial 2/10: Grass
  Trial 3/10: Grass
  Trial 4/10: Grass
  Trial 5/10: Grass
  Trial 6/10: Grass
  Trial 7/10: Grass.
  Trial 8/10: Grass
  Trial 9/10: Grass
  Trial 10/10: Grass


{'word': 'baskid',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Grass',
  'Grass',
  'Grass',
  'Grass',
  'Grass',
  'Grass',
  'Grass.',
  'Grass',
  'Grass',
  'Grass']}

In [20]:
run_experiment(word_data = baskid_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'baskid'...

Running 10 iterations for 'baskid' with gpt4 (elaborate prompt)...
  Trial 1/10: Grass
  Trial 2/10: Grass
  Trial 3/10: Grass
  Trial 4/10: Grass
  Trial 5/10: Grass
  Trial 6/10: Grass
  Trial 7/10: Grass
  Trial 8/10: Grass
  Trial 9/10: Grass
  Trial 10/10: Grass


{'word': 'baskid',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Grass',
  'Grass',
  'Grass',
  'Grass',
  'Grass',
  'Grass',
  'Grass',
  'Grass',
  'Grass',
  'Grass']}

In [21]:
merplin_data = create_word_data(word ='merplin', image_paths =  ['main_quine_file/quine_images/snow_images/snow.jpg', 
                                              'main_quine_file/quine_images/snow_images/artificial_snow.jpg',
                                             'main_quine_file/quine_images/snow_images/dirty_snow.jpg',
                                             'main_quine_file/quine_images/snow_images/snowflake.jpg',
                                             'main_quine_file/quine_images/snow_images/mountain_snow.jpg'])

run_experiment(word_data = merplin_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'merplin'...

Running 10 iterations for 'merplin' with gemini (minimal prompt)...
  Trial 1/10: Snow
  Trial 2/10: SNOW
  Trial 3/10: Snow
  Trial 4/10: SNOW
  Trial 5/10: SNOW
  Trial 6/10: SNOW
  Trial 7/10: SNOW
  Trial 8/10: SNOW
  Trial 9/10: SNOW
  Trial 10/10: SNOW


{'word': 'merplin',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Snow',
  'SNOW',
  'Snow',
  'SNOW',
  'SNOW',
  'SNOW',
  'SNOW',
  'SNOW',
  'SNOW',
  'SNOW']}

In [22]:
run_experiment(word_data = merplin_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'merplin'...

Running 10 iterations for 'merplin' with gemini (elaborate prompt)...
  Trial 1/10: Snow
  Trial 2/10: Snow
  Trial 3/10: Snow
  Trial 4/10: Snow
  Trial 5/10: Snow
  Trial 6/10: Snow
  Trial 7/10: Snow
  Trial 8/10: Snow
  Trial 9/10: Snow
  Trial 10/10: Snow


{'word': 'merplin',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Snow',
  'Snow',
  'Snow',
  'Snow',
  'Snow',
  'Snow',
  'Snow',
  'Snow',
  'Snow',
  'Snow']}

In [23]:
run_experiment(word_data = merplin_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'merplin'...

Running 10 iterations for 'merplin' with claude (minimal prompt)...
  Trial 1/10: I'm looking at these 5 winter images carefully:

1. A frost-covered tree in a snowy field
2. A snow-making machine spraying snow on a ski slope
3. Snow/ice buildup along a roadside
4. A detailed macro shot of a snowflake crystal
5. A snow-covered mountain ridge

The common theme across all images is **snow** in various forms - natural snow, artificial snow, ice crystals, and snow accumulation.

**SNOW**
  Trial 2/10: Looking at these 5 images, I can see:

1. A frost-covered tree in a snowy winter landscape
2. A snow-making machine spraying artificial snow on a ski slope
3. Piled snow (possibly artificial/man-made) along a roadside
4. A detailed snowflake crystal
5. A snow-covered mountain ridge

The common element across all images is **snow**, particularly in its various forms - natural snow, artificial snow, snow crystals, and snow accumulation.

**Snow**
  Trial 3/10

{'word': 'merplin',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ["I'm looking at these 5 winter images carefully:\n\n1. A frost-covered tree in a snowy field\n2. A snow-making machine spraying snow on a ski slope\n3. Snow/ice buildup along a roadside\n4. A detailed macro shot of a snowflake crystal\n5. A snow-covered mountain ridge\n\nThe common theme across all images is **snow** in various forms - natural snow, artificial snow, ice crystals, and snow accumulation.\n\n**SNOW**",
  'Looking at these 5 images, I can see:\n\n1. A frost-covered tree in a snowy winter landscape\n2. A snow-making machine spraying artificial snow on a ski slope\n3. Piled snow (possibly artificial/man-made) along a roadside\n4. A detailed snowflake crystal\n5. A snow-covered mountain ridge\n\nThe common element across all images is **snow**, particularly in its various forms - natural snow, artificial snow, snow crystals, and snow accumulation.\n\n**Snow**',
  'Looking at

In [24]:
run_experiment(word_data = merplin_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'merplin'...

Running 10 iterations for 'merplin' with claude (elaborate prompt)...
  Trial 1/10: Cold
  Trial 2/10: Cold
  Trial 3/10: Cold
  Trial 4/10: Cold
  Trial 5/10: Cold
  Trial 6/10: Cold
  Trial 7/10: Cold
  Trial 8/10: Cold
  Trial 9/10: Cold
  Trial 10/10: **Cold**


{'word': 'merplin',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Cold',
  'Cold',
  'Cold',
  'Cold',
  'Cold',
  'Cold',
  'Cold',
  'Cold',
  'Cold',
  '**Cold**']}

In [25]:
run_experiment(word_data = merplin_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'merplin'...

Running 10 iterations for 'merplin' with gpt4 (minimal prompt)...
  Trial 1/10: Snow
  Trial 2/10: Snow
  Trial 3/10: Snow
  Trial 4/10: Snow
  Trial 5/10: Snow.
  Trial 6/10: Snow
  Trial 7/10: Snow
  Trial 8/10: Snow
  Trial 9/10: Snow
  Trial 10/10: Snow


{'word': 'merplin',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Snow',
  'Snow',
  'Snow',
  'Snow',
  'Snow.',
  'Snow',
  'Snow',
  'Snow',
  'Snow',
  'Snow']}

In [26]:
run_experiment(word_data = merplin_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'merplin'...

Running 10 iterations for 'merplin' with gpt4 (elaborate prompt)...
  Trial 1/10: Snow
  Trial 2/10: Snow
  Trial 3/10: Snow
  Trial 4/10: Snow
  Trial 5/10: Snow
  Trial 6/10: Snow
  Trial 7/10: Snow
  Trial 8/10: Snow
  Trial 9/10: Snow
  Trial 10/10: Snow


{'word': 'merplin',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Snow',
  'Snow',
  'Snow',
  'Snow',
  'Snow',
  'Snow',
  'Snow',
  'Snow',
  'Snow',
  'Snow']}